# 5. Human-in-the-Loop — Pause, Approve, Resume

## Why

Some actions shouldn't run unattended: sending emails, issuing refunds, deleting data,
posting publicly. You want the AI to do the *work* and a human to give the *go-ahead*.

## How LangGraph does it: `interrupt()`

Inside any node, call `interrupt(payload)`. The graph **freezes right there**, persisting
all state via the checkpointer. The payload surfaces to your application. Minutes or days
later, you resume with `Command(resume=value)` — and that `value` becomes the *return value*
of the `interrupt()` call inside the node. Execution continues as if nothing happened.

Requirements: a **checkpointer** and a **thread_id** (that's how it knows what to resume).

## Real-life example: outreach email approval

This repo already has a job-application mailer — this is its missing safety layer.
Flow: AI drafts an outreach email → **human reviews** → approve (send) / give feedback
(AI revises, asks again) — a loop with a human inside it.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
# .env in this repo stores the key as GOOGLE_API_KEY_1 — normalize it
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY") or os.getenv("GOOGLE_API_KEY_1")

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite")
llm.invoke("Say 'ready' if you can hear me.").content

In [ ]:
from typing import TypedDict, Literal
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command


class OutreachState(TypedDict):
    company: str
    role: str
    draft: str
    feedback: str
    status: str

In [ ]:
def draft_email(state: OutreachState):
    feedback_part = ""
    if state.get("feedback"):
        feedback_part = f'''
A human reviewed your previous draft and said: "{state["feedback"]}"
Previous draft:
{state["draft"]}
Revise accordingly.'''
    prompt = f'''Write a short (under 120 words) outreach email applying for the
{state["role"]} role at {state["company"]}. Confident, specific, no fluff.{feedback_part}'''
    return {"draft": llm.invoke(prompt).content}


def human_review(state: OutreachState):
    # THE GRAPH STOPS HERE. The dict below is shown to the human.
    decision = interrupt({
        "draft": state["draft"],
        "question": "Approve this email? Reply {'action': 'approve'} or {'action': 'revise', 'feedback': '...'}",
    })
    if decision["action"] == "approve":
        return {"status": "approved"}
    return {"status": "needs_revision", "feedback": decision["feedback"]}


def send_email(state: OutreachState):
    # Real life: SMTP / Gmail API call goes here
    print(f">>> EMAIL SENT to {state['company']} <<<")
    return {"status": "sent"}


def route_after_review(state: OutreachState) -> Literal["send_email", "draft_email"]:
    return "send_email" if state["status"] == "approved" else "draft_email"


builder = StateGraph(OutreachState)
builder.add_node("draft_email", draft_email)
builder.add_node("human_review", human_review)
builder.add_node("send_email", send_email)

builder.add_edge(START, "draft_email")
builder.add_edge("draft_email", "human_review")
builder.add_conditional_edges("human_review", route_after_review)
builder.add_edge("send_email", END)

app = builder.compile(checkpointer=InMemorySaver())   # checkpointer is REQUIRED for interrupt

In [ ]:
# Visualize the graph (needs internet for mermaid rendering — safe to skip)
from IPython.display import Image, display

try:
    display(Image(app.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Could not render image, here is the mermaid source instead:\n")
    print(app.get_graph().draw_mermaid())

In [ ]:
config = {"configurable": {"thread_id": "application-langgraph-inc"}}

# Run until the graph hits interrupt() and freezes
result = app.invoke(
    {"company": "LangGraph Inc", "role": "AI Engineer", "draft": "", "feedback": "", "status": ""},
    config,
)

# The interrupt payload is what the human sees:
payload = result["__interrupt__"][0].value
print("DRAFT AWAITING APPROVAL:\n")
print(payload["draft"])

In [ ]:
# Human says: revise. Command(resume=...) is delivered as interrupt()'s return value.
result = app.invoke(
    Command(resume={"action": "revise", "feedback": "Mention my 3 years of Python experience and make it shorter."}),
    config,
)

# Graph looped: draft_email revised, then hit human_review's interrupt AGAIN
print("REVISED DRAFT AWAITING APPROVAL:\n")
print(result["__interrupt__"][0].value["draft"])

In [ ]:
# Human approves — graph resumes, routes to send_email, finishes
result = app.invoke(Command(resume={"action": "approve"}), config)
print("Final status:", result["status"])

## Key takeaways

- `interrupt(payload)` pauses; `Command(resume=value)` continues; `value` is what
  `interrupt()` returns inside the node. State survives the pause via the checkpointer.
- The pause can last **days** — with a database checkpointer, the process can even
  restart in between. This is how approval queues and "review dashboards" are built.
- Note the human is part of a **loop**: revise → re-review → approve.
- Same pattern: refund approvals, content moderation queues, contract review,
  any "AI proposes, human disposes" workflow.

**Next:** notebook 6 — multiple specialized agents working as a team.